In [9]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, f1_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\youss\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
df = pd.read_csv("spam.csv", encoding='latin-1')[['v1','v2']]
df.columns = ['label', 'message']
df['label'] = df['label'].map({'ham':0, 'spam':1})
df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [11]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{string.punctuation}]", "", text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df['clean_message'] = df['message'].apply(clean_text)

In [12]:
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(df['clean_message'])
y = df['label']

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:
models = {
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {"n_neighbors": [3,5,7]}
    },
    
    "Logistic Regression": {
        "model": LogisticRegression(max_iter=1000),
        "params": {"C": [0.1, 1, 10]}
    },
    
    "Naive Bayes": {
        "model": MultinomialNB(),
        "params": {"alpha": [0.5, 1.0, 1.5]}
    }
}

In [15]:
results = {}

for name, mp in models.items():
    grid = GridSearchCV(mp['model'], mp['params'], cv=5, scoring='f1')
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results[name] = f1
    
    print(f"\n{name}")
    print(f"Best Params : {grid.best_params_}")
    print(f"Accuracy    : {acc:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"F1 Score    : {f1:.4f}")


KNN
Best Params : {'n_neighbors': 3}
Accuracy    : 0.9202
Precision   : 1.0000
F1 Score    : 0.5782

Logistic Regression
Best Params : {'C': 10}
Accuracy    : 0.9695
Precision   : 0.9603
F1 Score    : 0.8768

Naive Bayes
Best Params : {'alpha': 0.5}
Accuracy    : 0.9776
Precision   : 0.9921
F1 Score    : 0.9097


In [16]:
best_model_name = max(results, key=results.get)
print("Best Model:", best_model_name)

Best Model: Naive Bayes
